In [1]:
# import os.path as osp
# from mmengine import Config, DictAction
# # from mmdet.datasets import build_dataset
# from mmengine.runner import Runner


import argparse
import os
import os.path as osp


import mmcv
import mmengine
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules
from mmengine.config import Config, DictAction
from mmengine.registry import RUNNERS
from mmengine.runner import Runner

In [2]:
register_all_modules()

In [3]:
import repvit

In [4]:
config = "configs/mask_rcnn_repvit_m1_1_fpn_1x_coco.py"
cfg = Config.fromfile(config)
cfg.work_dir = osp.join("./work_dirs", osp.splitext(osp.basename(config))[0])

In [5]:
runner = Runner.from_cfg(cfg)

05/04 23:28:07 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: darwin
    Python: 3.10.17 | packaged by conda-forge | (main, Apr 10 2025, 22:23:34) [Clang 18.1.8 ]
    CUDA available: False
    MUSA available: False
    numpy_random_seed: 1598082351
    GCC: Apple clang version 17.0.0 (clang-1700.3.5.51)
    PyTorch: 1.13.0
    PyTorch compiling details: PyTorch built with:
  - GCC 4.2
  - C++ Version: 201402
  - clang 14.0.0
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: NO AVX
  - Build settings: BLAS_INFO=accelerate, BUILD_TYPE=Release, CXX_COMPILER=/Applications/Xcode_14.0.1.app/Contents/Developer/Toolchains/XcodeDefault.xctoolchain/usr/bin/c++, CXX_FLAGS= -Wno-deprecated -fvisibility-inlines-hidden -Wno-deprecated-declarations -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINETO -DLIBKINETO_NOCUPTI -DUSE_PYTORCH_QNNPACK -DUSE_XNNPACK -DUSE_PYTORCH_METAL_EXPORT -DSYMBOLICATE_

In [2]:
cfg = Config.fromfile("configs/mask_rcnn_repvit_m1_1_fpn_1x_coco.py")
datasets = [build_dataset(cfg.data.train)]
datasets.append(build_dataset(cfg.data.val))

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [3]:
for idx in range(len(datasets[1])):
    data_ann = datasets[1].get_ann_info(idx)
    print(data_ann["masks"])


[None, None, None, None, None]
[None, None, None]
[None, None, None, None, None, None, None]
[None, None, None, None, None]
[None, None, None, None, None, None]
[None, None, None, None]
[None, None, None]
[None, None, None, None]
[None, None, None, None, None]
[None, None, None, None, None]
[None]
[None, None, None, None]
[None, None, None]
[None, None, None, None]
[None, None, None, None, None, None, None]
[None, None, None, None, None]
[None, None, None]
[None, None, None, None, None, None]
[None, None, None, None]
[None, None, None, None, None, None]
[None, None, None, None]
[None, None, None, None, None, None, None]
[None, None, None, None]
[None, None, None]
[None, None, None]
[None, None, None]
[None, None, None, None]
[None, None, None]
[None, None, None, None, None]
[None, None, None]
[None, None, None]
[None, None, None]
[None, None, None, None, None]
[None, None, None, None]
[None, None, None, None]
[None, None, None, None, None]
[None, None, None]
[None, None, None, None, No

In [4]:
data_ann

{'bboxes': array([[ 148.  ,  321.  ,  181.97,  368.13],
        [ 192.  ,  328.  ,  220.55,  382.18],
        [ 240.  ,  293.  ,  351.33,  456.85],
        [   0.  ,  342.  ,  196.74, 1049.86]], dtype=float32),
 'labels': array([ 9,  8,  9, 11]),
 'bboxes_ignore': array([], shape=(0, 4), dtype=float32),
 'masks': None,
 'seg_map': 'subwayy1_1_mp4-71_jpg.rf.dac04baa5fb1b74eb34ba3f853c534f7.png'}

In [5]:
datasets

[
 CocoDataset Train dataset with number of images 1816, and instance counts: 
 +-----------------------+-------+----------------+-------+-------------------------+-------+----------------------+-------+----------------+-------+
 | category              | count | category       | count | category                | count | category             | count | category       | count |
 +-----------------------+-------+----------------+-------+-------------------------+-------+----------------------+-------+----------------+-------+
 | 0 [boots-powerup]     | 20    | 1 [boxtrain]   | 1081  | 2 [jump-barrier]        | 326   | 3 [jump-electric]    | 9     | 4 [jump-hedge] | 36    |
 | 5 [jump-roll-barrier] | 623   | 6 [jump-trash] | 76    | 7 [ledge]               | 109   | 8 [pogstick-powerup] | 86    | 9 [ramp]       | 987   |
 | 10 [roll-barrier]     | 256   | 11 [train]     | 2155  | 12 [undercover-barrier] | 178   | 13 [wall]            | 902   | -1 background  | 0     |
 +-------------------

In [13]:
import json
import argparse
from collections import Counter


def count_instances_by_category_id(annotation_file):
    with open(annotation_file, "r") as f:
        coco_data = json.load(f)
    print(f'categories: {[cat["name"] for cat in coco_data["categories"]]}')

    annotations = coco_data.get("annotations", [])

    category_counter = Counter()
    for ann in annotations:
        category_id = ann.get("category_id")
        if category_id is not None:
            category_counter[category_id] += 1

    for category_id, count in sorted(category_counter.items()):
        print(f"Category ID {category_id}: {count} instances")

In [14]:
count_instances_by_category_id("data/coco/annotations/instances_train2017.json")

categories: ['boots-powerup', 'boxtrain', 'jump-barrier', 'jump-electric', 'jump-hedge', 'jump-roll-barrier', 'jump-trash', 'ledge', 'pogstick-powerup', 'ramp', 'roll-barrier', 'train', 'undercover-barrier', 'wall']
Category ID 1: 20 instances
Category ID 2: 1081 instances
Category ID 3: 326 instances
Category ID 4: 9 instances
Category ID 5: 36 instances
Category ID 6: 623 instances
Category ID 7: 76 instances
Category ID 8: 109 instances
Category ID 9: 86 instances
Category ID 10: 987 instances
Category ID 11: 256 instances
Category ID 12: 2155 instances
Category ID 13: 178 instances
Category ID 14: 902 instances


In [15]:
count_instances_by_category_id("data/coco/annotations/instances_val2017.json")

categories: ['boots-powerup', 'boxtrain', 'jump-barrier', 'jump-electric', 'jump-hedge', 'jump-roll-barrier', 'jump-trash', 'ledge', 'pogstick-powerup', 'ramp', 'roll-barrier', 'train', 'undercover-barrier', 'wall']
Category ID 1: 4 instances
Category ID 2: 293 instances
Category ID 3: 91 instances
Category ID 4: 5 instances
Category ID 5: 8 instances
Category ID 6: 140 instances
Category ID 7: 20 instances
Category ID 8: 26 instances
Category ID 9: 32 instances
Category ID 10: 236 instances
Category ID 11: 53 instances
Category ID 12: 636 instances
Category ID 13: 37 instances
Category ID 14: 241 instances
